# Auditorium AHU pipeline walkthrough

This notebook is the readable demonstration layer for the auditorium study. It loads artefacts produced by the tested Python pipeline and creates report-ready Matplotlib figures. It does not reimplement auditing, cleaning, feature engineering or pattern discovery.

The walkthrough covers:

1. raw-data audit and label balance;
2. the Phase 2 prepared table and data-quality policy;
3. Phase 4 operating-mode availability and cleaning outcomes;
4. Phase 5 feature families and shortcut-audit findings; and
5. normal-operation profiles, PCA and operating-mode patterns.

Interpretation is deliberately cautious: these are descriptive and data-quality results, not evidence that a detector generalises to unseen buildings.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT = Path.cwd()
if not (ROOT / "configs/auditorium.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "configs/auditorium.yaml").exists(), "Open this notebook from the repository or its notebooks/ directory."

AUDIT_DIR = ROOT / "results/audit"
AUDITORIUM_DIR = ROOT / "data/processed/auditorium"
PHASE4_DIR = AUDITORIUM_DIR / "phase4"
PHASE5_DIR = AUDITORIUM_DIR / "phase5"
PATTERN_DIR = AUDITORIUM_DIR / "patterns"

COLORS = {
    "normal": "#2a6fbb",
    "fault": "#c94c4c",
    "heating": "#d97706",
    "cooling": "#168aad",
    "accent": "#6b4c9a",
}

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def require_artifacts(*paths):
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing generated artefacts. Run the pipeline commands first:\n"
            + "\n".join(f"- {path}" for path in missing)
        )

print(f"Repository: {ROOT}")

## Reproducibility gate

The notebook expects the command-line pipeline to have been run first. The status table makes missing stages visible rather than silently recomputing them inside the notebook.

In [ ]:
artifacts = {
    "raw audit": AUDIT_DIR / "audit_manifest.json",
    "prepared table": AUDITORIUM_DIR / "auditorium_prepared.csv",
    "Phase 4 cleaning": PHASE4_DIR / "phase4_manifest.json",
    "Phase 5 features": PHASE5_DIR / "phase5_manifest.json",
    "pattern discovery": PATTERN_DIR / "pattern_manifest.json",
}
status = pd.DataFrame({
    "stage": list(artifacts),
    "path": [str(path.relative_to(ROOT)) for path in artifacts.values()],
    "available": [path.exists() for path in artifacts.values()],
})
display(status)
if not status["available"].all():
    display(Markdown("**Action required:** complete the missing pipeline stages before running the remaining cells."))
else:
    display(Markdown("All expected pipeline artefacts are available."))

## 1. Audit: what is in the dataset?

The first figure separates the source labels from the binary detection target. The second shows missingness by measured signal. Missing heating and cooling values are expected to be mode-dependent in this dataset, so missingness is investigated rather than automatically treated as a fault.

In [ ]:
require_artifacts(
    AUDIT_DIR / "audit_label_counts.csv",
    AUDIT_DIR / "audit_missingness.csv",
)
labels = pd.read_csv(AUDIT_DIR / "audit_label_counts.csv")
labels = labels.query("building == 'auditorium' and dimension == 'overall'").copy()
labels["target"] = np.where(labels["label"].eq("Normal condition"), "normal", "fault")
labels = labels.sort_values("count", ascending=True)

missingness = pd.read_csv(AUDIT_DIR / "audit_missingness.csv")
missingness = missingness.query("building == 'auditorium'").copy()
missingness = missingness.sort_values("missing_rate")

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
axes[0].barh(labels["label"], labels["count"], color=[COLORS[t] for t in labels["target"]])
axes[0].set_title("Auditorium source-label support")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("Source label")
for index, row in labels.reset_index(drop=True).iterrows():
    axes[0].text(row["count"], index, f"  {int(row['count']):,}", va="center", fontsize=9)
axes[1].barh(missingness["column"], 100 * missingness["missing_rate"], color=COLORS["accent"])
axes[1].set_title("Missingness by measured signal")
axes[1].set_xlabel("Missing rows (%)")
axes[1].set_ylabel("Source column")
axes[1].set_xlim(0, 100)
plt.show()

In [ ]:
audit = read_json(AUDIT_DIR / "audit_manifest.json")
prep = read_json(AUDITORIUM_DIR / "auditorium_preparation_manifest.json")
headline = pd.DataFrame({
    "Audit measure": [
        "Source rows", "AHUs", "Time range", "Duplicate AHU/time keys",
        "Time-gap rows", "Normal rows", "Fault rows",
    ],
    "Value": [
        f"{prep['source_row_count']:,}",
        f"{prep['ahu_count']:,}",
        f"{prep['timestamp_minimum']} to {prep['timestamp_maximum']}",
        audit.get("buildings", {}).get("auditorium", {}).get("duplicate_ahu_timestamp_keys", "see audit manifest"),
        f"{prep['time_gap_rows']:,}",
        f"{prep['target_binary_counts']['normal']:,}",
        f"{prep['target_binary_counts']['fault']:,}",
    ],
})
display(headline)

## 2. Cleaning and operating modes

Phase 4 treats heating/cooling missingness as structural subsystem availability. The plot below shows the operating-mode coverage and the second-audit counts. No values are silently filled in the command-line cleaning stage; any later imputation must be fitted within each training fold.

In [ ]:
require_artifacts(PHASE4_DIR / "phase4_manifest.json", PHASE4_DIR / "phase4_audit.csv")
phase4 = read_json(PHASE4_DIR / "phase4_manifest.json")
mode_counts = pd.Series(phase4["summary"]["operating_mode_counts"], name="rows").sort_values()
audit4 = pd.read_csv(PHASE4_DIR / "phase4_audit.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].bar(mode_counts.index, mode_counts.values, color=[COLORS.get(mode, COLORS["accent"]) for mode in mode_counts.index])
axes[0].set_title("Operating-mode coverage")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=20)
for index, value in enumerate(mode_counts.values):
    axes[0].text(index, value, f"{value:,}", ha="center", va="bottom")

audit4_plot = audit4.set_index("sensor")[["structural_missing_count", "short_gap_eligible_count", "imputed_count"]]
audit4_plot.plot.barh(ax=axes[1], color=["#8ecae6", "#ffb703", "#2a9d8f"])
axes[1].set_title("Phase 4 missingness handling")
axes[1].set_xlabel("Rows")
axes[1].set_ylabel("Sensor")
axes[1].legend(["Structural missing", "Eligible short gap", "Imputed"], fontsize=8)
plt.show()

display(pd.DataFrame([phase4["summary"]]).T.rename(columns={0: "value"}))

## 3. Feature engineering and shortcut audit

The feature lineage shows how the model-ready table is constructed. The shortcut audit is a review tool: it does not remove features, but it identifies univariate value separation or missingness separation that must be discussed before modelling.

In [ ]:
require_artifacts(PHASE5_DIR / "phase5_manifest.json", PHASE5_DIR / "feature_lineage.csv", PHASE5_DIR / "shortcut_audit.csv")
lineage = pd.read_csv(PHASE5_DIR / "feature_lineage.csv")
shortcut = pd.read_csv(PHASE5_DIR / "shortcut_audit.csv")
family_counts = lineage.groupby("family").size().sort_values()
review = shortcut.query("review_required == True").copy()
review["max_separation"] = review[["missing_rate_gap", "value_auc"]].apply(
    lambda row: max(abs(row.get("missing_rate_gap", np.nan)), abs(row.get("value_auc", 0.5) - 0.5)), axis=1
)
review = review.sort_values("max_separation", ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
axes[0].barh(family_counts.index, family_counts.values, color=COLORS["accent"])
axes[0].set_title("Generated features by family")
axes[0].set_xlabel("Feature count")
axes[0].set_ylabel("Feature family")

if review.empty:
    axes[1].text(0.5, 0.5, "No shortcut flags", ha="center", va="center")
    axes[1].set_axis_off()
else:
    plot_review = review.sort_values("max_separation")
    axes[1].barh(plot_review["feature"], plot_review["max_separation"], color=COLORS["fault"])
    axes[1].axvline(0.25, color="#555555", linestyle="--", linewidth=1, label="review threshold")
    axes[1].set_title("Strongest shortcut-audit flags")
    axes[1].set_xlabel("Separation measure")
    axes[1].legend(fontsize=8)
plt.show()

display(pd.DataFrame({"feature_count": [len(lineage)], "review_flags": [int(shortcut["review_required"].sum())]}))
display(review[["variant", "feature", "family", "missing_rate_gap", "value_auc", "review_required"]])

## 4. Normal-operation patterns

These plots use the saved normal-only pattern artefacts. They show how the AHU population behaves by hour, how the two operating modes differ, and whether normal rows form visible low-dimensional clusters. PCA is a visual summary; its mode-specific median filling is not a training imputation method.

In [ ]:
require_artifacts(
    PATTERN_DIR / "pattern_manifest.json",
    PATTERN_DIR / "hourly_profile.csv",
    PATTERN_DIR / "monthly_profile.csv",
    PATTERN_DIR / "pca_coordinates.csv",
)
pattern_manifest = read_json(PATTERN_DIR / "pattern_manifest.json")
hourly = pd.read_csv(PATTERN_DIR / "hourly_profile.csv")
monthly = pd.read_csv(PATTERN_DIR / "monthly_profile.csv")
pca = pd.read_csv(PATTERN_DIR / "pca_coordinates.csv")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for mode, color in [("heating", COLORS["heating"]), ("cooling", COLORS["cooling"])]:
    for sensor, linestyle in [("Supply air temperature", "-"), ("Return temperature", "--")]:
        subset = hourly.query("operating_mode == @mode and sensor == @sensor")
        if not subset.empty:
            axes[0].plot(subset["hour"], subset["mean"], color=color, linestyle=linestyle, marker="o", label=f"{mode}: {sensor}")
axes[0].set_title("Normal hourly temperature profiles")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Dataset value")
axes[0].set_xticks(range(0, 24, 3))
axes[0].legend(fontsize=8, ncol=2)

for mode, color in [("heating", COLORS["heating"]), ("cooling", COLORS["cooling"])]:
    subset = monthly.query("operating_mode == @mode and sensor == 'Supply air temperature'")
    if not subset.empty:
        axes[1].plot(subset["month"], subset["mean"], color=color, marker="o", label=mode)
axes[1].set_title("Normal monthly supply-air profile")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Dataset value")
axes[1].set_xticks(range(1, 13))
axes[1].legend()
plt.show()

display(pd.DataFrame([pattern_manifest["summary"]]).T.rename(columns={0: "value"}))

In [ ]:
plot_pca = pca.sample(min(12000, len(pca)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for axis, mode in zip(axes, ["heating", "cooling"]):
    subset = plot_pca.query("operating_mode == @mode")
    if subset.empty:
        axis.text(0.5, 0.5, f"No {mode} PCA rows", ha="center", va="center")
        axis.set_axis_off()
        continue
    for ahu, group in subset.groupby("ahu", sort=True):
        axis.scatter(group["pc1"], group["pc2"], s=7, alpha=0.22, label=ahu)
    axis.set_title(f"Normal-operation PCA: {mode}")
    axis.set_xlabel("PC1")
    axis.set_ylabel("PC2")
    if subset["ahu"].nunique() <= 15:
        axis.legend(fontsize=7, ncol=2, title="AHU")
plt.show()

## Interpretation

Use the figures to support three defensible observations: (1) the dataset contains multiple source fault classes and substantial mode-dependent missingness; (2) the cleaning policy retains structural missingness rather than fabricating readings; and (3) normal operation varies by mode, time and AHU. These plots are evidence for feature and evaluation design, not a substitute for grouped or building-held-out model evaluation.